# Not using

In [ ]:
import json
import logging
import torch
from torch.utils.data import TensorDataset
from transformers import AutoTokenizer, InputExample, InputFeatures

# For generator model
def load_for_generator(data_file, local_rank, max_seq_length, tokenizer, evaluate=False,
                            input_label="question", target_label='inter_Step'):

    #Local rank stuff
    if local_rank == -1:
        local_rank = 0

    nr_examples = 0
    examples = []
    labels = []

    with open(data_file, 'r') as f:
        data = json.load(f)

    for i, pair in enumerate(data):
        # Determine input text based on available keys and evaluation mode.
        if input_label not in pair or not pair[input_label]:
            if not evaluate:
                text_a = pair["question"] + pair["evidence"]
            else:
                text_a = pair["question"]
        else:
            if not evaluate:
                text_a = pair["question"] + pair["evidence"]
            else:
                text_a = pair[input_label]

        # Process the target label.
        # Here, we use the raw label text directly.
        label = str(pair[target_label]).replace("'", "")
        labels.append(label)

        guid = str(i)
        ex = InputExample(guid=guid, text_a=text_a, text_b=None, label=label)
        examples.append(ex)
        nr_examples += 1

    logging.info("Number of examples: %d", nr_examples)

    # Tokenize the input texts.
    tokenized_inputs = tokenizer.batch_encode_plus(
        [ex.text_a for ex in examples],
        padding="longest",  # Or 'max_length' if you prefer fixed-length padding.
        max_length=max_seq_length,
        pad_to_max_length=True,  # Note: this parameter is deprecated in recent versions.
        truncation=True,
        return_tensors="pt",
    )

    # Tokenize the target labels.
    tokenized_targets = tokenizer.batch_encode_plus(
        [ex.label for ex in examples],
        padding="longest",  # Or 'max_length'.
        max_length=max_seq_length,
        pad_to_max_length=True,
        truncation=True,
        return_tensors="pt",
    )


    dataset = TensorDataset(
        tokenized_inputs['input_ids'], tokenized_inputs['attention_mask'],
        tokenized_targets['input_ids'], tokenized_targets['attention_mask']
    )

    return dataset, labels


In [ ]:
import json
import torch
from transformers import AutoTokenizer

# Set these variables as needed
data_file = "test_inter.json"
local_rank = -1  # assuming non-distributed training for testing
max_seq_length = 64  # adjust as needed
evaluate = False

# Load a pre-trained tokenizer (using T5 as an example)
tokenizer = AutoTokenizer.from_pretrained("google/t5-v1_1-small", use_fast=False)

# Load the dataset using your function
dataset, labels = load_for_generator(
    data_file=data_file,
    local_rank=local_rank,
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    evaluate=evaluate,
    input_label="question",
    target_label="inter_Step"
)

# Print some details to verify tokenization and dataset shapes
print("Number of examples:", len(dataset))
input_ids, input_masks, target_ids, target_masks = dataset.tensors
print("Input IDs shape:", input_ids.shape)
print("Input Masks shape:", input_masks.shape)
print("Target IDs shape:", target_ids.shape)
print("Target Masks shape:", target_masks.shape)
print("Labels:", labels)

# Optionally, inspect the first example:
print("\nFirst example (input tokens):", input_ids[0])
print("First example (target tokens):", target_ids[0])

# Install model, packages, etc.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth
    !pip install -U bitsandbytes

In [ ]:
!pip uninstall bitsandbytes
!pip install -U bitsandbytes


Found existing installation: bitsandbytes 0.45.4
Uninstalling bitsandbytes-0.45.4:
  Would remove:
    /usr/local/lib/python3.11/dist-packages/bitsandbytes-0.45.4.dist-info/*
    /usr/local/lib/python3.11/dist-packages/bitsandbytes/*
Proceed (Y/n)? t
Your response ('t') was not one of the expected responses: y, n, 
Proceed (Y/n)? y
  Successfully uninstalled bitsandbytes-0.45.4
  Using cached bitsandbytes-0.45.4-py3-none-manylinux_2_24_x86_64.whl.metadata (5.0 kB)
Using cached bitsandbytes-0.45.4-py3-none-manylinux_2_24_x86_64.whl (76.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2025.3.19 requires tyro, which is not installed.
unsloth 2025.3.19 requires protobuf<4.0.0, but you have protobuf 5.29.4 which is incompatible.
unsloth 2025.3.19 requires trl!=0.15.0,!=0.9.0,!=0.9.1,!=0.9.2,!=0.9.3,<=0.15.2,>=0.7.9, but you have trl 0.16.0 which is inco

# Define the model

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    model_name = "unsloth/Qwen2.5-14B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)
tokenizer.truncation_side = "left"

==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Lora

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Load the hf dataset and add db_schema to the dataset that we use for training

In [ ]:
import pandas as pd
from datasets import load_dataset

hf_df = pd.read_parquet("hf://datasets/xu3kev/BIRD-SQL-data-train/data/train-00000-of-00001-fe8894d41b7815be.parquet")

print(hf_df.head())

db_schema_mapping = hf_df.set_index("db_id")["schema"].to_dict()
# Load the JSON file
dataset = load_dataset("json", data_files="ground_truth_inters.json", field=None)

print(dataset.column_names)

#add new column as db_schema
def add_schema(example):
  example["db_schema"] = db_schema_mapping.get(example["db_id"], None)
  return example

# Use the map function to add the column to the Dataset
dataset = dataset.map(add_schema)

print(dataset.column_names)

dataset = dataset['train']

               db_id                                           question  \
0        video_games  List down at least five publishers of the game...   
1   retail_complains  What is the full address of the customers who,...   
2             disney    How many movies did Wolfgang Reitherman direct?   
3         legislator              How many legislators are not senator?   
4  food_inspection_2  How many inspections done by Lisa Tillman ende...   

                                            evidence  \
0  publishers refers to publisher_name; number of...   
1  full address = address_1, address_2; received ...   
2  Wolfgang Reitherman refers director = 'Wolfgan...   
3               not senator refers to class is null;   
4  the result of "Out of Business" refers to resu...   

                                                 SQL  \
0  SELECT T.publisher_name FROM ( SELECT DISTINCT...   
1  SELECT T1.address_1, T1.address_2 FROM client ...   
2  SELECT COUNT(name) FROM director WHERE di

Test

In [ ]:
print(dataset.column_names)

#print dataset's first 5 rows
print(dataset["train"][:5])

{'train': ['query_id', 'feedback_id', 'db_id', 'question', 'evidence', 'SQL', 'inter_step', 'false_inter_step', 'feedback_line', 'operation', 'db_schema']}
{'query_id': [0, 1, 2, 3, 4], 'feedback_id': ['0', '0', '0', '0', '0'], 'db_id': ['movie_platform', 'movie_platform', 'movie_platform', 'movie_platform', 'movie_platform'], 'question': ['Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.', 'State the most popular movie? When was it released and who is the director for the movie?', 'What is the name of the longest movie title? When was it released?', 'Name the movie with the most ratings.', 'What is the average number of Mubi users who love movies directed by Stanley Kubrick?'], 'evidence': ['released in the year 1945 refers to movie_release_year = 1945;', 'most popular movie refers to MAX(movie_popularity); when it was released refers to movie_release_year; director for the movie refers to director_name;', 'longest movie title refe

# Prompt

In [ ]:
alpaca_prompt = """Objective: You are an intermediate step generator. Your task is to determine which tables, columns, and values to use in order to generate an SQL query with respect to a given question. You must analyze the given query and given database schema. By analyzing both, you must extract the database entities related to the given question. You will have feedback based on your generated intermediate step.
Instructions:
1. Analyze the database schema and understand the structure of the database. Determine all the tables, their columns, and relations between tables.
2. Read the Question Carefully: Understand the primary focus and specific details of the question. Look for any named entities (such as organizations, locations, etc.), technical terms, and other phrases that encapsulate important aspects of the inquiry.
3. Analyze The Hint If It is Given: The hint is designed to direct attention toward certain elements relevant to answering the question. Extract any keywords, phrases, or named entities that could provide further clarity of direction in formulating an answer.
4. Generate an Intermediate Step: By analyzing the given database schema, question, and hint determine all the tables, those tables columns, and values to use in order to generate an SQL to retrieve the asked data. Generate intermediate steps with the same structure given in Examples below.

### Database Schema
{}

### Question
{}

### Hint
{}

### Intermediate Step Answer:
OUTPUT:
{}

"""
def formatting_prompts_func(examples):
    # Extract the 'question' and 'inter_Step' fields
    db_ids = examples['db_id']
    questions = examples["question"]
    hints = examples['evidence']
    inter_steps = examples["inter_step"]
    db_schemas = examples["db_schema"]

    texts = []
    for db_schema, question, hint, inter_step in zip(db_schemas, questions, hints, inter_steps):
        # Format the prompt as desired. Here, we simply combine the question and inter_Step.
        text = alpaca_prompt.format(db_schema, question, hint, inter_step)
        texts.append(text)

    return {"text": texts}
pass

#Summarize the dataset
print(dataset.column_names)

split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

train_dataset = dataset.remove_columns(['query_id', 'feedback_id',  'SQL', 'false_inter_step', 'feedback_line', 'operation'])
test_dataset = dataset.remove_columns(['query_id', 'feedback_id',  'SQL', 'false_inter_step', 'feedback_line', 'operation'])

# Apply the formatting function on the dataset
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

print(train_dataset.column_names)

['query_id', 'feedback_id', 'db_id', 'question', 'evidence', 'SQL', 'inter_step', 'false_inter_step', 'feedback_line', 'operation', 'db_schema']
['db_id', 'question', 'evidence', 'inter_step', 'db_schema', 'text']


dataset's token count

In [ ]:
token_lengths = []
print(len(train_dataset))
count = 0
for example in train_dataset:
    text = example["text"]
    # Tokenize without truncation
    tokens = tokenizer(text, truncation=False)
    if len(tokens["input_ids"]) > 2048:
        count += 1
    token_lengths.append(len(tokens["input_ids"]))

max_length = max(token_lengths)
average_length = sum(token_lengths) / len(token_lengths)

print(f"Max tokens: {max_length}")
print(f"Average tokens: {average_length}")
print(count)

# Trainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset=train_dataset,  # use the actual split
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        #max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9428 [00:00<?, ? examples/s]

In [ ]:
train_dataset['text'][1]

'Objective: You are an intermediate step generator. Your task is to determine which tables, columns, and values to use in order to generate an SQL query with respect to a given question. You must analyze the given query and given database schema. By analyzing both, you must extract the database entities related to the given question. You will have feedback based on your generated intermediate step.\nInstructions:\n1. Analyze the database schema and understand the structure of the database. Determine all the tables, their columns, and relations between tables.\n2. Read the Question Carefully: Understand the primary focus and specific details of the question. Look for any named entities (such as organizations, locations, etc.), technical terms, and other phrases that encapsulate important aspects of the inquiry.\n3. Analyze The Hint If It is Given: The hint is designed to direct attention toward certain elements relevant to answering the question. Extract any keywords, phrases, or named 

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,428 | Num Epochs = 3 | Total steps = 7,071
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 137,625,600/14,000,000,000 (0.98% trained)


Step,Training Loss
1,0.001000
2,0.000800


KeyboardInterrupt: 

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

319.4591 seconds used for training.
5.32 minutes used for training.
Peak reserved memory = 12.594 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 85.435 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        train_dataset['db_schema'][1],
        train_dataset['question'][1],
        train_dataset['evidence'][1],
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    temperature=0.9,  # Slightly higher temperature might encourage diversity
    top_p=0.8,        # Adjusting top-p can also affect diversity
    do_sample=True
)




Objective: You are an intermediate step generator. Your task is to determine which tables, columns, and values to use in order to generate an SQL query with respect to a given question. You must analyze the given query and given database schema. By analyzing both, you must extract the database entities related to the given question. You will have feedback based on your generated intermediate step.
Instructions:
1. Analyze the database schema and understand the structure of the database. Determine all the tables, their columns, and relations between tables.
2. Read the Question Carefully: Understand the primary focus and specific details of the question. Look for any named entities (such as organizations, locations, etc.), technical terms, and other phrases that encapsulate important aspects of the inquiry.
3. Analyze The Hint If It is Given: The hint is designed to direct attention toward certain elements relevant to answering the question. Extract any keywords, phrases, or named entit

KeyboardInterrupt: 